# Phase 1 — StreamFlix Data Cleaning & Quality Check

Prepared from the supplied StreamFlix project brief, data dictionary, schema, and six CSV files.

## 1. Load all six tables
The project brief requires profiling row/column counts, data types, nulls, duplicates, dates, outliers, referential integrity, and business-rule checks.

In [2]:
import pandas as pd, numpy as np
from pathlib import Path

DATA_DIR = Path(".")  # change if the CSV files are stored elsewhere

subscribers = pd.read_csv(DATA_DIR/"subscribers.csv", parse_dates=["signup_date","churn_date"])
titles = pd.read_csv(DATA_DIR/"titles.csv", parse_dates=["date_added","license_expiry"])
watch_history = pd.read_csv(DATA_DIR/"watch_history.csv", parse_dates=["watch_date"])
ratings = pd.read_csv(DATA_DIR/"ratings.csv", parse_dates=["rating_date"])
reviews = pd.read_csv(DATA_DIR/"reviews.csv", parse_dates=["review_date"])
watchlist = pd.read_csv(DATA_DIR/"watchlist.csv", parse_dates=["added_date"])

tables = {
    "subscribers": subscribers,
    "titles": titles,
    "watch_history": watch_history,
    "ratings": ratings,
    "reviews": reviews,
    "watchlist": watchlist
}

In [3]:
profile = pd.DataFrame([{
    'table':name, 'rows':len(df), 'columns':df.shape[1]
} for name,df in tables.items()])
profile

,table,rows,columns
0,subscribers,15000,14
1,titles,9000,21
2,watch_history,650000,10
3,ratings,130000,5
4,reviews,110000,7
5,watchlist,65000,5


In [4]:
for name, df in tables.items():
    print(f'\n{name}:')
    display(df.dtypes.to_frame('dtype'))


subscribers:


,dtype
subscriber_id,str
signup_date,datetime64[us]
country,str
region,str
age,int64
gender,str
plan_type,str
monthly_price_usd,float64
household_size,int64
primary_device,str



titles:


,dtype
title_id,str
title_name,str
type,str
primary_genre,str
country,str
language,str
release_year,int64
date_added,datetime64[us]
maturity_rating,str
seasons,int64



watch_history:


...

,dtype
rating_id,int64
subscriber_id,str
title_id,str
rating,int64
rating_date,datetime64[us]



reviews:


,dtype
review_id,int64
subscriber_id,str
title_id,str
review_text,str
sentiment,str
helpful_votes,int64
review_date,datetime64[us]



watchlist:


,dtype
watchlist_id,int64
subscriber_id,str
title_id,str
added_date,datetime64[us]
watched,bool


## 2. Missing-value report

In [5]:
missing = []
for name, df in tables.items():
    for col in df.columns:
        n = int(df[col].isna().sum())
        missing.append([name,col,n,round(n/len(df)*100,3)])
missing_df = pd.DataFrame(missing, columns=['table','column','null_count','null_pct'])
missing_df[missing_df.null_count>0].sort_values(['table','null_count'],ascending=[True,False])

,table,column,null_count,null_pct
13,subscribers,churn_date,11199,74.660
32,titles,license_expiry,1966,21.844


## 3. Duplicate and key checks

In [6]:
checks = pd.DataFrame([
 ['subscribers','subscriber_id',subscribers['subscriber_id'].duplicated().sum()],
 ['titles','title_id',titles['title_id'].duplicated().sum()],
 ['watch_history','watch_id',watch_history['watch_id'].duplicated().sum()],
 ['ratings','rating_id',ratings['rating_id'].duplicated().sum()],
 ['reviews','review_id',reviews['review_id'].duplicated().sum()],
 ['watchlist','watchlist_id',watchlist['watchlist_id'].duplicated().sum()],
], columns=['table','key','duplicates'])
checks

,table,key,duplicates
0,subscribers,subscriber_id,0
1,titles,title_id,0
2,watch_history,watch_id,0
3,ratings,rating_id,0
4,reviews,review_id,0
5,watchlist,watchlist_id,0


## 4. Data-type and business-rule validation

In [7]:
# Date fields are parsed as datetime; numeric fields are checked below.
watch_history['duration_ratio_pct'] = watch_history['watch_duration_min']/watch_history['content_duration_min']*100
watch_history['completion_abs_diff'] = (watch_history['completion_pct']-watch_history['duration_ratio_pct']).abs()
print('Watch sessions longer than content:', (watch_history.watch_duration_min>watch_history.content_duration_min).sum())
print('Negative watch durations:', (watch_history.watch_duration_min<0).sum())
print('Completion within 1 percentage point:', (watch_history.completion_abs_diff<=1).mean()*100, '%')
print('Invalid review sentiments:', (~reviews.sentiment.isin(['Positive','Neutral','Negative'])).sum())
print('Churn date <= signup date for inactive:', ((~subscribers.is_active) & subscribers.churn_date.notna() & (subscribers.churn_date<=subscribers.signup_date)).sum())
print('Active subscribers with churn date:', (subscribers.is_active & subscribers.churn_date.notna()).sum())

Watch sessions longer than content: 0
Negative watch durations: 0
Completion within 1 percentage point: 100.0 %
Invalid review sentiments: 0
Churn date <= signup date for inactive: 0
Active subscribers with churn date: 0


## 5. Referential integrity

In [8]:
ri = pd.DataFrame([
 ['watch_history','subscriber_id',(~watch_history.subscriber_id.isin(subscribers.subscriber_id)).sum()],
 ['watch_history','title_id',(~watch_history.title_id.isin(titles.title_id)).sum()],
 ['ratings','subscriber_id',(~ratings.subscriber_id.isin(subscribers.subscriber_id)).sum()],
 ['ratings','title_id',(~ratings.title_id.isin(titles.title_id)).sum()],
 ['reviews','subscriber_id',(~reviews.subscriber_id.isin(subscribers.subscriber_id)).sum()],
 ['reviews','title_id',(~reviews.title_id.isin(titles.title_id)).sum()],
 ['watchlist','subscriber_id',(~watchlist.subscriber_id.isin(subscribers.subscriber_id)).sum()],
 ['watchlist','title_id',(~watchlist.title_id.isin(titles.title_id)).sum()],
],columns=['table','foreign_key','orphan_count'])
ri

,table,foreign_key,orphan_count
0,watch_history,subscriber_id,0
1,watch_history,title_id,0
2,ratings,subscriber_id,0
3,ratings,title_id,0
4,reviews,subscriber_id,0
5,reviews,title_id,0
6,watchlist,subscriber_id,0
7,watchlist,title_id,0


## 6. Data Quality conclusion
All checks should be reviewed before analysis. In this supplied dataset, the quality checks found no duplicate primary keys, no orphan foreign keys, no invalid sentiments, no duration-over-content sessions, and completion percentage is consistent with duration/content within 1 percentage point for all sessions. Nulls are expected in churn_date for active subscribers and license_expiry for Originals.